In [1]:
!pip install biopython py3Dmol pandas scipy requests

import os
import re
import requests
import numpy as np
import pandas as pd
import py3Dmol
from Bio.SVDSuperimposer import SVDSuperimposer
from IPython.display import display, HTML
import warnings
warnings.filterwarnings("ignore")

# --- 1. Helper functions from original script for fetching and parsing ---
_DNA = {'DA', 'DC', 'DG', 'DT', 'DU', 'DI'}
_RNA_MOD = {
    'PSU': 'U', 'H2U': 'U', '5MU': 'U', '4SU': 'U', '5MC': 'C', 'OMC': 'C',
    '1MA': 'A', '6MZ': 'A', 'MA6': 'A', 'A2M': 'A', '2MG': 'G', '1MG': 'G',
    '7MG': 'G', 'M2G': 'G', 'OMG': 'G', 'YG': 'G', 'G7M': 'G', 'UR3': 'U',
    'OMU': 'U', '5BU': 'U', 'CBV': 'C', 'CCC': 'C', 'GDP': 'G', 'GTP': 'G',
}

def _res_letter(resname):
    r = resname.strip().upper()
    if r in _DNA: return None
    if r in ('A', 'C', 'G', 'U'): return r
    if r in _RNA_MOD: return _RNA_MOD[r]
    return 'N'

def parse_entities(all_sequences_str):
    entities = []
    for entry in str(all_sequences_str).strip().split('>'):
        if not entry.strip(): continue
        parts = entry.split('|')
        n_chains = 0
        for p in parts:
            if 'Chain' in p:
                auth = re.findall(r'\[auth\s+([A-Za-z0-9]+)\]', p)
                if auth: n_chains = len(auth)
                else:
                    m = re.search(r'Chains?\s+([A-Za-z0-9,\s]+)', p)
                    if m: n_chains = len([c for c in re.split(r'[,\s]+', m.group(1)) if c])
                break
        if n_chains == 0: n_chains = 1
        seq = re.sub(r'[^A-Za-z]', '', parts[-1]).upper()
        if not seq: continue
        is_rna = all(c in 'ACGUN' for c in seq)
        entities.append({'seq': seq, 'n_chains': n_chains, 'is_rna': is_rna})
    return entities

def rna_segments(all_sequences_str, sequence):
    segs = []
    for e in parse_entities(all_sequences_str):
        if e['is_rna']: segs.extend([e['seq']] * e['n_chains'])
    if ''.join(segs) != str(sequence): segs = [str(sequence)]
    return segs

def fetch_reference(pdb_id, cache_dir="cif_cache"):
    os.makedirs(cache_dir, exist_ok=True)
    for ext, url in ((".cif", f"https://files.rcsb.org/download/{pdb_id}.cif"),
                     (".pdb", f"https://files.rcsb.org/download/{pdb_id}.pdb")):
        path = os.path.join(cache_dir, pdb_id + ext)
        if os.path.exists(path) and os.path.getsize(path) > 0: return path
        try:
            r = requests.get(url, timeout=60)
            if r.status_code == 200 and len(r.content) > 0:
                with open(path, "wb") as f: f.write(r.content)
                return path
        except: pass
    return None

def parse_reference_rna_chains(path):
    from Bio.PDB import MMCIFParser, PDBParser
    parser = MMCIFParser(QUIET=True) if path.endswith(".cif") else PDBParser(QUIET=True)
    structure = parser.get_structure("ref", path)
    model = next(iter(structure))
    chains = {}
    for chain in model:
        letters, coords = [], []
        for res in chain:
            if not res.has_id("C1'"): continue
            letter = _res_letter(res.get_resname())
            if letter is None: continue
            letters.append(letter)
            coords.append(res["C1'"].get_coord())
        if len(letters) >= 3:
            chains[chain.id] = {'seq': ''.join(letters), 'coords': np.asarray(coords, dtype=float)}
    return chains

def _align_map(pred_seq, ref_seq):
    from Bio.Align import PairwiseAligner
    aln = PairwiseAligner()
    aln.mode = 'global'
    aln.match_score, aln.mismatch_score = 2.0, -1.0
    aln.open_gap_score, aln.extend_gap_score = -2.0, -0.5
    a = aln.align(pred_seq, ref_seq)[0]
    pairs = []
    for (ps, pe), (rs, re_) in zip(a.aligned[0], a.aligned[1]):
        for k in range(pe - ps): pairs.append((ps + k, rs + k))
    return pairs

def build_correspondence(all_sequences, sequence, ref_chains):
    segs = rna_segments(all_sequences, sequence)
    avail = dict(ref_chains)
    pred_idx, ref_xyz = [], []
    offset = 0
    for seg in segs:
        best_id, best_pairs, best_score = None, None, -1
        for cid, info in avail.items():
            pairs = _align_map(seg, info['seq'])
            score = len(pairs) - 0.001 * abs(len(seg) - len(info['seq']))
            if score > best_score:
                best_id, best_pairs, best_score = cid, pairs, score
        if best_id is not None and best_pairs:
            coords = avail[best_id]['coords']
            for (pp, rp) in best_pairs:
                pred_idx.append(offset + pp)
                ref_xyz.append(coords[rp])
            del avail[best_id]
        offset += len(seg)
    if not pred_idx: return None, None
    return np.asarray(pred_idx, dtype=int), np.asarray(ref_xyz, dtype=float)

def get_predicted_models(sub_df, target_id, n_res):
    rows = sub_df[sub_df["ID"].str.rsplit("_", n=1).str[0] == target_id].copy()
    rows["ridx"] = rows["ID"].str.rsplit("_", n=1).str[1].astype(int)
    rows = rows.sort_values("ridx")
    models = []
    for m in range(1, 6):
        models.append(rows[[f"x_{m}", f"y_{m}", f"z_{m}"]].to_numpy(dtype=float))
    return models

def coords_to_pdb_str(coords, seq="A"):
    # Create a simple PDB string from coordinates for py3Dmol
    seq = seq * len(coords) if len(seq) == 1 else seq
    pdb_lines = []
    for i, ((x, y, z), ch) in enumerate(zip(coords, seq), 1):
        pdb_lines.append(f"ATOM  {i:>5}  C1' {ch:>3} A{i:>4}    {x:>8.3f}{y:>8.3f}{z:>8.3f}  1.00  0.00           C")
    for i in range(1, len(coords)):
        pdb_lines.append(f"CONECT{i:>5}{i+1:>5}")
    pdb_lines.append("END")
    return "\n".join(pdb_lines)

# --- 2. Main Visualization Logic ---
def plot_overlays_from_csv(sub_path, test_path, model_idx=1):
    sub = pd.read_csv(sub_path)
    test = pd.read_csv(test_path)
    
    for _, row in test.iterrows():
        tid = str(row["target_id"])
        seq = str(row["sequence"])
        
        # 1. Fetch Reference
        ref_path = fetch_reference(tid)
        if not ref_path:
            print(f"⚠️ {tid}: Could not fetch reference.")
            continue
            
        try:
            ref_chains = parse_reference_rna_chains(ref_path)
        except Exception as e:
            print(f"⚠️ {tid}: parse error: {e}")
            continue
            
        # 2. Build Correspondence
        pred_idx, ref_xyz = build_correspondence(row["all_sequences"], seq, ref_chains)
        if pred_idx is None:
            print(f"⚠️ {tid}: No correspondence found.")
            continue
            
        # 3. Get Predicted Model (defaulting to model 1 for overlay)
        models = get_predicted_models(sub, tid, len(seq))
        if not models or len(models[0]) == 0:
            print(f"⚠️ {tid}: No predictions found in submission.")
            continue
            
        pred_coords = models[model_idx - 1]
        if pred_coords.shape[0] <= max(pred_idx):
            continue
            
        pred_sub = pred_coords[pred_idx]
        
        # 4. Align Predicted to Reference using Python (SVDSuperimposer)
        sup = SVDSuperimposer()
        sup.set(ref_xyz, pred_sub)
        sup.run()
        rot, tran = sup.get_rotran()
        
        # Apply rotation and translation to the *full* predicted model for visual overlay
        pred_superposed = np.dot(pred_coords, rot) + tran
        
        # 5. Generate PDB strings in memory
        pred_pdb = coords_to_pdb_str(pred_superposed, seq)
        # For the reference, we just plot the matched residues to align with our script's logic
        ref_pdb = coords_to_pdb_str(ref_xyz, "A" * len(ref_xyz))
        
        # 6. Render with py3Dmol
        print(f"\n▶ {tid} — Displaying Model {model_idx} Overlay")
        view = py3Dmol.view(width=900, height=550)

        view.addModel(ref_pdb, "pdb")
        view.setStyle({"model": 0}, {
            "stick":  {"color": "#1f77b4", "radius": 0.35, "opacity": 0.9},
            "sphere": {"color": "#1f77b4", "scale": 0.3}
        })

        view.addModel(pred_pdb, "pdb")
        view.setStyle({"model": 1}, {
            "stick":  {"color": "#d62728", "radius": 0.35, "opacity": 0.9},
            "sphere": {"color": "#d62728", "scale": 0.3}
        })

        view.setBackgroundColor("0xf7f7f7")
        view.zoomTo()

        display(HTML(f"""
        <div style="font-family: sans-serif; margin-bottom: 6px;">
          <b>{tid}</b><br>
          <span style="color:#1f77b4;">&#9679;</span> experimental (aligned reference subset) &nbsp;&nbsp;
          <span style="color:#d62728;">&#9679;</span> predicted (Model {model_idx})
        </div>
        """))
        view.show()

# Run the visualizer directly from your CSVs (update paths as needed)
plot_overlays_from_csv(
    sub_path="/kaggle/input/datasets/khushvaghela/test-submission/final_submission.csv",
    test_path="/kaggle/input/datasets/khushvaghela/test-submission/final_test_sequences.csv",
    model_idx=1  # Choose which of the 5 models to visualize
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 53.0 MB/s eta 0:00:00

▶ 8ZNQ — Displaying Model 1 Overlay


3Dmol.js failed to load for some reason. Please check your browser console for error messages.


▶ 9IWF — Displaying Model 1 Overlay


3Dmol.js failed to load for some reason. Please check your browser console for error messages.


▶ 9JGM — Displaying Model 1 Overlay


...

3Dmol.js failed to load for some reason. Please check your browser console for error messages.


▶ 9LJN — Displaying Model 1 Overlay


3Dmol.js failed to load for some reason. Please check your browser console for error messages.


▶ 9ZCC — Displaying Model 1 Overlay


3Dmol.js failed to load for some reason. Please check your browser console for error messages.